# The page and its bars

Session 2, *The source and the notes*: the check at the end of the setup, and part 6.

This notebook prepares our source for the edition. It downloads the scan, copies page 8 into a PDF of its own, draws that page as an image, and lists which bars are on which system. Next session we link every bar to its place on this image.

Keep `schumann-edition-inputs.zip` in the same folder as the notebook (`get_notebooks.py` puts it there). It holds our starting MEI file and the settings for this page; the scan is downloaded when you run the notebook.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. If something goes wrong, the message appears under the cell.

## Setup

The first cell makes sure the packages are installed; after the setup they already are, so it finishes quickly. The second cell defines helper functions; it is collapsed because you do not need to read it. The third checks that everything is there: it prints `CAMAT: 0.2.8` and `pypdf: 6.10.0`. During the setup, you stop there.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" pypdf==6.10.0 verovio==6.3.0

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Schumann witness preparation; notation/settings live in a separate ZIP."""

from pathlib import Path
import hashlib
import io
import json
import urllib.request
import xml.etree.ElementTree as ET
import zipfile


def pdf_file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def find_schumann_pdf_inputs(start=None):
    start = Path(start or Path.cwd()).resolve()
    name = "schumann-edition-inputs.zip"
    for folder in (start, *start.parents):
        for candidate in (folder / name, folder / "notebooks" / name,
                          folder / "sessions/02-building-an-edition/notebooks" / name):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Copy {name} beside this notebook, or run get_notebooks.py to get both files.")


def _preserve_or_write(path, data):
    """Keep existing matching inputs; never replace an edited or different file."""
    path = Path(path)
    if path.exists():
        if path.read_bytes() != data:
            raise FileExistsError(f"Preserving different file: {path}. Choose a new PROJECT folder.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("xb") as output:
            output.write(data)


def _preserve_json(path, value):
    _preserve_or_write(path, (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode())


def load_schumann_pdf_inputs(bundle, project):
    """Unpack a checked copy of the starter, credits and preparation settings."""
    bundle, project = Path(bundle).resolve(), Path(project).resolve()
    required = {"starter.mei", "LICENSE", "sources.json", "settings.json"}
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("manifest.json"))
        if set(manifest["files_sha256"]) != required or set(archive.namelist()) != required | {"manifest.json"}:
            raise ValueError("Unexpected Schumann input bundle contents.")
        files = {name: archive.read(name) for name in required}
    for name, data in files.items():
        if hashlib.sha256(data).hexdigest() != manifest["files_sha256"][name]:
            raise ValueError(f"Input hash mismatch: {name}")
    folder = project / "inputs" / pdf_file_sha256(bundle)[:12]
    for name, data in files.items():
        _preserve_or_write(folder / name, data)
    return folder, json.loads(files["settings.json"]), json.loads(files["sources.json"])


def prepare_schumann_pdfs(project, settings, sources, *, local_original=None, allow_download=True):
    """Check the frozen volume, then extract original page 8 without changing it.

    pypdf is pinned because even a visually identical derivative can have
    different PDF bytes. CAMAT renders only this separately identified extract.
    """
    import pypdf
    if pypdf.__version__ != settings["pypdf_version"]:
        raise RuntimeError(f"This notebook needs pypdf {settings['pypdf_version']}, but {pypdf.__version__} is installed. "
                           "Run the first cell, restart the kernel, and run the notebook again.")
    folder = Path(project).resolve() / "sources"
    original = folder / settings["original_filename"]
    excerpt = folder / settings["excerpt_filename"]
    witness = sources["printed_witness"]
    expected_original = witness["sha256"]
    if original.exists():
        if pdf_file_sha256(original) != expected_original:
            raise ValueError(f"{original} is not the scan the course uses. Move it away and run the cell again.")
    else:
        if local_original is not None:
            data = Path(local_original).read_bytes()
        elif allow_download:
            request = urllib.request.Request(witness["download_url"], headers={"User-Agent": "camat-schumann-seminar"})
            try:
                with urllib.request.urlopen(request, timeout=90) as response:
                    data = response.read(witness["bytes"] + 1)
            except OSError as problem:
                raise RuntimeError(f"Could not download the scan from IMSLP ({problem}). Check the internet "
                                   f"connection and run the cell again, or put a copy of the file here: {original}") from None
        else:
            raise FileNotFoundError("Supply LOCAL_ORIGINAL or enable ALLOW_SOURCE_DOWNLOAD.")
        if len(data) != witness["bytes"] or hashlib.sha256(data).hexdigest() != expected_original:
            raise ValueError("The download is not the scan the course uses. Run the cell again later.")
        _preserve_or_write(original, data)
    reader = pypdf.PdfReader(original)
    if len(reader.pages) != witness["page_count"]:
        raise ValueError("Unexpected original PDF page count.")
    page = reader.pages[settings["original_pdf_page"] - 1]
    writer, memory = pypdf.PdfWriter(), io.BytesIO()
    writer.add_page(page)
    writer.add_metadata(sources["classroom_pdf"]["pdf_metadata"])
    writer.write(memory)
    data = memory.getvalue()
    if hashlib.sha256(data).hexdigest() != sources["classroom_pdf"]["sha256"]:
        raise ValueError("Extracted PDF differs from the frozen one-page derivative; review the recipe.")
    _preserve_or_write(excerpt, data)
    reread = pypdf.PdfReader(excerpt)
    if len(reread.pages) != 1 or pdf_file_sha256(original) != expected_original:
        raise ValueError("Extraction integrity check failed.")
    record = {
        "original_pdf": str(original.relative_to(Path(project).resolve())),
        "original_sha256": expected_original, "original_page_count": len(reader.pages),
        "excerpt_pdf": str(excerpt.relative_to(Path(project).resolve())),
        "excerpt_sha256": pdf_file_sha256(excerpt), "excerpt_page_count": 1,
        "original_pdf_page": settings["original_pdf_page"], "excerpt_pdf_page": 1,
        "printed_page": settings["printed_page"], "pypdf": pypdf.__version__,
        "transformation": sources["classroom_pdf"]["transformation"],
        "pdf_metadata": sources["classroom_pdf"]["pdf_metadata"],
        "media_box_points": list(map(float, page.mediabox)),
        "crop_box_points": list(map(float, page.cropbox)), "rotation": page.rotation,
    }
    _preserve_json(folder / "page-mapping.json", record)
    return original, excerpt, record


def schumann_body_measures(mei_file):
    ns = {"m": "http://www.music-encoding.org/ns/mei"}
    body = ET.parse(mei_file).getroot().find("m:music/m:body", ns)
    if body is None:
        raise ValueError("No music/body found.")
    measures = body.findall(".//m:measure", ns)
    labels = [m.get("n") for m in measures]
    if labels != list(map(str, range(1, 22))):
        raise ValueError(f"This workshop expects the whole-piece carrier, labels 1–21. Found {labels}")
    return measures


def print_bar_table(mei_file, settings):
    """Bars as musicians count them, beside the MEI numbers, systems and IDs."""
    by_label = {m.get("n"): m for m in schumann_body_measures(mei_file)}
    print(f"{'Bar':<8}{'MEI n':<7}{'System':<8}xml:id")
    for system, labels in enumerate(settings["system_mei_labels"], 1):
        for label in labels:
            bar = "pickup" if label == "1" else str(int(label) - 1)
            print(f"{bar:<8}{label:<7}{system:<8}{by_label[label].get('{http://www.w3.org/XML/1998/namespace}id')}")


def save_schumann_page_checkpoint(project, mei_file, settings, pdf_record, pages_json):
    """Verify the technical map and save the handoff to the measure-box block."""
    project, pages_json = Path(project).resolve(), Path(pages_json).resolve()
    manifest = json.loads(pages_json.read_text())
    if manifest["page_count"] != 1 or manifest["dpi"] != 300:
        raise ValueError("Expected one rendered page at 300 dpi.")
    if manifest["pdf_sha256"] != pdf_record["excerpt_sha256"]:
        raise ValueError("Page images do not belong to the frozen excerpt PDF.")
    page = manifest["pages"][0]
    if page["role"] != "music" or page["printed_page"] != "8" or page["pdf_page_number"] != 1:
        raise ValueError("Check the excerpt-page/printed-page description.")
    systems = [s["measures"] if isinstance(s, dict) else s for s in page["systems"]]
    if systems != settings["system_mei_labels"]:
        raise ValueError("The systems in part 4 do not match the print. Compare where each system starts "
                         "with the image, correct the line in part 4, run that cell again, then this one.")
    if [page["width"], page["height"]] != settings["expected_image_pixels"]:
        raise ValueError("Image geometry differs; review before drawing boxes.")
    image = pages_json.parent / page["image"]
    if pdf_file_sha256(image) != page["sha256"]:
        raise ValueError("Page image changed after rendering.")
    by_label = {m.get("n"): m for m in schumann_body_measures(mei_file)}
    identifiers = [m.get("{http://www.w3.org/XML/1998/namespace}id") for m in by_label.values()]
    if None in identifiers or len(set(identifiers)) != len(identifiers):
        raise ValueError("Measure IDs must be present and unique before facsimile linking.")
    rows = []
    for number, labels in enumerate(systems, 1):
        for label in labels:
            rows.append({"mei_label": label, "measure_xml_id": by_label[label].get("{http://www.w3.org/XML/1998/namespace}id"),
                         "conventional_bar": "pickup" if label == "1" else str(int(label) - 1),
                         "system": number, "original_pdf_page": 8, "excerpt_pdf_page": 1, "printed_page": "8",
                         "core_editing_passage": label in settings["reviewed_mei_labels"]})
    _preserve_json(project / "edition/measure-page-map.json", rows)
    record = {"stage": "page preparation; measure boxes and facsimile links are not created yet",
              "mei_input": str(Path(mei_file).resolve().relative_to(project)), "mei_sha256": pdf_file_sha256(mei_file),
              "pages_json": str(pages_json.relative_to(project)), "pages_json_sha256": pdf_file_sha256(pages_json),
              "image_sha256": page["sha256"], "image_pixels": [page["width"], page["height"]],
              "renderer": manifest["renderer"], "original_to_excerpt_to_printed": [8, 1, "8"],
              "body_measures": len(rows), "systems": len(systems),
              "scope": settings["scope"], "next_step": "Review one measure box per MEI measure, then merge facsimile links."}
    _preserve_json(project / "page-preparation-checkpoint.json", record)
    return rows, record

In [2]:
import importlib.metadata
from IPython.display import display, Image
from camat.pdf_facsimile import check_poppler, parse_page_layout, check_page_layout
from camat.pdf_facsimile_guided import make_page_images, describe_pages

check_poppler()
print("CAMAT:", importlib.metadata.version("camat"))
print("pypdf:", importlib.metadata.version("pypdf"))

CAMAT: 0.2.8
pypdf: 6.10.0


## 1. Download the scan and copy page 8

Our source is [IMSLP file 51556](https://imslp.org/wiki/Special:ImagefromIndex/51556): Schumann's *Album für die Jugend* in Clara Schumann's edition (Breitkopf & Härtel), scanned by the Bavarian State Library. The PDF has 50 pages; our piece, No. 10, is on page 8.

The next cell

1. unpacks our starting files into the folder `schumann_edition_work` next to this notebook;
2. downloads the scan (9 MB) into `schumann_edition_work/sources`, unless `02_schumann_omr.ipynb` already did, and keeps it unchanged;
3. copies page 8 into a one-page PDF. In that file it is page 1, but its printed page number stays 8.

It also checks that the scan is exactly the file we prepared the course with, and that the copy of page 8 comes out the same as ours.

<details><summary>How can a notebook tell that two files are the same?</summary>

It computes a *fingerprint* (a SHA-256 hash) from every byte of the file: a code of 64 letters and digits. If a single byte differs, the code is completely different. The notebook compares the code of your download with the one we recorded.

</details>

In [3]:
PROJECT = Path.cwd() / "schumann_edition_work"
INPUT_BUNDLE = find_schumann_pdf_inputs()
INPUTS, SETTINGS, SOURCES = load_schumann_pdf_inputs(INPUT_BUNDLE, PROJECT)
MEI_FILE = INPUTS / "starter.mei"
OUTPUT_FOLDER = PROJECT / "edition"
MEI_HASH_BEFORE = pdf_file_sha256(MEI_FILE)

ORIGINAL_PDF, PDF_FILE, PDF_RECORD = prepare_schumann_pdfs(PROJECT, SETTINGS, SOURCES)
print("Starting MEI file:", MEI_FILE)
print("The scan, unchanged:", ORIGINAL_PDF)
print("Page 8 alone:", PDF_FILE)

Starting MEI file: <notebook folder>/schumann_edition_work/inputs/b02b5197b383/starter.mei
The scan, unchanged: <notebook folder>/schumann_edition_work/sources/imslp-51556-schumann-op68.pdf
Page 8 alone: <notebook folder>/schumann_edition_work/sources/schumann-op68-no10-source-page.pdf


## 2. The page as an image

CAMAT draws the one-page PDF as a PNG image at 300 dpi (dots per inch): 4167 × 5617 pixels. It writes down how it made the image in `edition/pages/pages.json`. Next session we draw a box around every bar on this image, so the image must stay exactly as it is. The notebook shows it smaller; the boxes will refer to the full size. CAMAT's message asks which pages show music: we answer that in part 4.

In [4]:
make_page_images(PDF_FILE, OUTPUT_FOLDER, download_example=False)

Turning the PDF pages into pictures (300 dpi). This takes a moment ...


✓ 1 pages, saved in schumann_edition_work/edition/pages


Which PDF pages show music? You need that for step 2.


In [5]:
PAGES_JSON = OUTPUT_FOLDER / "pages/pages.json"
manifest = json.loads(PAGES_JSON.read_text())
page = manifest["pages"][0]
assert manifest["page_count"] == 1 and manifest["dpi"] == 300
assert manifest["pdf_sha256"] == PDF_RECORD["excerpt_sha256"]
assert [page["width"], page["height"]] == SETTINGS["expected_image_pixels"]
print("Image:", PAGES_JSON.parent / page["image"])
print("Size:", page["width"], "×", page["height"], "pixels at", manifest["dpi"], "dpi")

Image: <notebook folder>/schumann_edition_work/edition/pages/page-0001.png
Size: 4167 × 5617 pixels at 300 dpi


In [6]:
display(Image(filename=str(PAGES_JSON.parent / page["image"]), width=850))

## 3. Bars and systems

A system is the group of staves played together, joined by a brace at the left. Our page has five systems. The print has no bar numbers; we count as musicians do: the pickup, then bar 1 for the first full bar.

The MEI file numbers its bars (`n`) from the pickup, so its numbers are one higher than ours. The table also shows each bar's `xml:id`, its name in the file. Our edition covers the pickup and bars 1–8: the first two systems.

In [7]:
print_bar_table(MEI_FILE, SETTINGS)

Bar     MEI n  System  xml:id
pickup  1      1       d1e9
1       2      1       d1e74
2       3      1       d1e352
3       4      1       d1e648
4       5      1       d1e968
5       6      2       d1e1231
6       7      2       d1e1509
7       8      2       d1e1805
8       9      2       d1e2125
9       10     3       d1e2402
10      11     3       d1e2690
11      12     3       d1e2973
12      13     3       d1e3252
13      14     4       d1e3589
14      15     4       d1e3953
15      16     4       d1e4265
16      17     4       d1e4550
17      18     5       d1e4833
18      19     5       d1e5112
19      20     5       d1e5449
20      21     5       d1e5813


## 4. Describe the page for CAMAT

CAMAT needs to know which bars are on which system. We write that as one line per page:

`PDF page | printed page | MEI numbers per system`

The PDF page is 1 (our one-page PDF), the printed page is 8. The systems are separated by commas; `1-5` means the MEI bars 1 to 5, that is the pickup and bars 1–4. Check the line against the image above, then run the cell. CAMAT checks it against the MEI file and answers with the MEI numbers. The *step 4* in its last line is CAMAT's own next step, the linking: we do that next session.

In [8]:
MUSIC_PAGES = """
1 | 8 | 1-5, 6-9, 10-13, 14-17, 18-21
"""

describe_pages(MUSIC_PAGES, MEI_FILE, OUTPUT_FOLDER)

PDF page | printed page | content
       1 |            8 | music: 5 systems, bars 1–21

✓ Your description matches the MEI: all 21 bars, in the same order.
  (The movement in the MEI has no number; step 4 numbers it 1 in the new file.)


### Extension: what the check can see, and what it cannot

CAMAT checks that every bar of the MEI file is listed once and in order. It cannot see where a system ends on the page: that is our job. The next cell tries two wrong lines, without saving them. The first leaves out the pickup and is rejected. The second starts the second system one bar too early, and is accepted.

In [9]:
missing_pickup = "1 | 8 | 2-5, 6-9, 10-13, 14-17, 18-21"
try:
    check_page_layout(parse_page_layout(missing_pickup, page_count=1), MEI_FILE)
except ValueError as problem:
    print("Without the pickup: rejected.", problem)
else:
    raise AssertionError("The line without the pickup was accepted.")

wrong_break = "1 | 8 | 1-4, 5-9, 10-13, 14-17, 18-21"
check_page_layout(parse_page_layout(wrong_break, page_count=1), MEI_FILE)
print("With a wrong system break: accepted. Only a look at the image shows the mistake.")

Without the pickup: rejected. Your description and the MEI do not have the same bars.
  In the MEI but not in your description: 1
The MEI has 21 bars: 1–21.
With a wrong system break: accepted. Only a look at the image shows the mistake.


## 5. Save for next session

The last cell checks everything once more and saves the table of bars, systems and IDs as `edition/measure-page-map.json`. Keep the folder `schumann_edition_work` as it is: next session continues there.

In [10]:
rows, checkpoint = save_schumann_page_checkpoint(PROJECT, MEI_FILE, SETTINGS, PDF_RECORD, PAGES_JSON)
assert pdf_file_sha256(MEI_FILE) == MEI_HASH_BEFORE
assert pdf_file_sha256(ORIGINAL_PDF) == SOURCES["printed_witness"]["sha256"]
print("Saved:", len(rows), "bars (the pickup and bars 1–20) on", checkpoint["systems"], "systems.")
print("The scan and the starting MEI file are unchanged.")
print("Next session: a box around each bar, linked to the MEI file.")

Saved: 21 bars (the pickup and bars 1–20) on 5 systems.
The scan and the starting MEI file are unchanged.
Next session: a box around each bar, linked to the MEI file.


Links: [CAMAT's guide to linking a PDF](https://github.com/egorpol/camat_v2/blob/v0.2.8/notebooks/mei_pdf_facsimile_guided.ipynb), the [IMSLP scan](https://imslp.org/wiki/Special:ImagefromIndex/51556), and the [MEI Guidelines on facsimiles](https://music-encoding.org/guidelines/v5/content/facsimilesrecordings.html). Our starting MEI file is based on Maja Hartwig's encoding in the MEI Sample Collection; her credits and the licence are in `schumann_edition_work/inputs`.